# Quality of Care Indicators — Report

Résumé des indicateurs de qualité des soins (taux de test, traitement, létalité, etc.) par district et année. Les cartes sont générées par le notebook de calcul et sauvegardées dans `reporting/outputs`.

## Configuration

In [ ]:
ROOT_PATH <- "~/workspace"
CODE_PATH <- file.path(ROOT_PATH, "code")
CONFIG_PATH <- file.path(ROOT_PATH, "configuration")
DATA_PATH <- file.path(ROOT_PATH, "data", "dhis2")
QOC_PATH <- file.path(DATA_PATH, "quality_of_care")
PLOTS_PATH <- file.path(ROOT_PATH, "pipelines", "snt_quality_of_care", "reporting", "outputs")

source(file.path(CODE_PATH, "snt_utils.r"))
required_packages <- c("dplyr", "tidyr", "arrow", "knitr", "glue", "jsonlite")
install_and_load(required_packages)

In [ ]:
config_json <- tryCatch(
  jsonlite::fromJSON(file.path(CONFIG_PATH, "SNT_config.json")),
  error = function(e) stop("Error loading config: ", conditionMessage(e))
)
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE
COUNTRY_NAME <- config_json$SNT_CONFIG$COUNTRY_NAME
dataset_qoc <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_QUALITY_OF_CARE

## Chargement des données QoC

In [ ]:
qoc_file <- paste0(COUNTRY_CODE, "_quality_of_care.parquet")
qoc_data <- NULL
if (!is.null(dataset_qoc) && dataset_qoc != "") {
  qoc_data <- tryCatch(
    get_latest_dataset_file_in_memory(dataset_qoc, qoc_file),
    error = function(e) NULL
  )
}
if (is.null(qoc_data)) {
  path_local <- file.path(QOC_PATH, qoc_file)
  if (file.exists(path_local)) {
    qoc_data <- arrow::read_parquet(path_local)
    log_msg(glue("Loaded QoC data from {path_local}"))
  }
}
if (is.null(qoc_data)) {
  stop(glue("No Quality of Care data found. Run the snt_quality_of_care pipeline first."))
}
print(dim(qoc_data))
head(qoc_data, 5)

## Résumé par année

In [ ]:
rate_cols <- c("testing_rate", "treatment_rate", "case_fatality_rate", "prop_adm_malaria", "prop_malaria_deaths")
rate_cols <- intersect(rate_cols, names(qoc_data))
if (length(rate_cols) > 0 && "YEAR" %in% names(qoc_data)) {
  summary_year <- qoc_data %>%
    group_by(YEAR) %>%
    summarise(across(any_of(rate_cols), ~ mean(.x, na.rm = TRUE)), .groups = "drop")
  knitr::kable(summary_year, digits = 3)
} else {
  print("No rate columns or YEAR found.")
}

## Cartes

Les cartes (PNG) par indicateur et année sont générées par le notebook de calcul et enregistrées dans `pipelines/snt_quality_of_care/reporting/outputs/`.

In [ ]:
if (dir.exists(PLOTS_PATH)) {
  list.files(PLOTS_PATH, pattern = "\.png$")
} else {
  cat("Outputs folder not found:", PLOTS_PATH)
}